In [ ]:
import os
import sys

import collections
import xarray as xr
xr.set_options(keep_attrs=True)
import netCDF4 as nc
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
import metpy.calc as mp
from metpy.units import units
from scipy.stats import ttest_ind, ttest_rel
from datetime import datetime

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

In [ ]:
def windSpd(u,v):
   windSpd=np.sqrt(u**2 + v**2)
   return windSpd

In [ ]:
### +++ LOAD CLIMATE DATA +++ ###

diags=['u1000','v1000']
dat={ 'merra2':{}, 'max.global':{} }

# number of years to omit 
omit_yrs = 30
n_omit = omit_yrs * 12
tmax = 350 * 12

# calculate annual mean fields
for key in dat.keys():
    for varn in diags:
        if key=='merra2':
            dat[key][varn] = xr.open_dataset(f'obs.merra2.{varn}.nc')[varn].mean(dim='time')
        else:
            dat[key][varn] = xr.open_dataset(f'aogcm.{key}.{varn}.nc')[varn].isel(time=slice(n_omit,tmax)).mean(dim='time')

# calculate wind speed from u and v components
for key in dat.keys()
    dat[key]['spd']=windSpd(dat[key]['u1000'],dat[key]['v1000'])

In [ ]:
text_kw={'weight':'bold', 'ha':'center', 'va':'center', 'color':'black', 'size':14}
# colormap
cmap=sns.diverging_palette(20, 220, as_cmap=True)
vmin=-5
vmax=5
levels=np.linspace(vmin, vmax, 11)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
# vector specs
skip_n=2
w=0.003
scalef=20
key_length=5
# map specs
lon=dat['max.global']['spd'].lon
lon=dat['max.global']['spd'].lat
proj=ccrs.PlateCarree()
trans=ccrs.PlateCarree()
map_bnds=[-180., 120., -60., 80.]

fig,ax=plt.subplots(nrows=1, ncols=1, figsize=(9, 6), layout='constrained', subplot_kw={'projection': proj})

ax.text(-30,86,'MAX$_{global}$ $-$ OBS', rotation=0, **text_kw)

# mean wind speed field difference
ax.pcolormesh(lon, lat, (dat['max.global']['spd']-dat['merra2']['spd']),
              norm=norm, cmap=cmap, transform=trans)

# wind speed vector differences
q1=ax.quiver(lon[::skip_n], lat[::skip_n],
             (dat['max.global']['u']-dat['merra2']['u'])[::skip_n,::skip_n],
             (dat['max.global']['v']-dat['merra2']['v'])[::skip_n,::skip_n],
             color='k', width=w, scale=scalef, scale_units='inches', units='height',
             transform=trans, zorder=100)
qk=ax.quiverkey(q1, .97, 1.07, key_length, 'MAX$_{global}$ bias', labelcolor='k', labelpos='W')

# merra climatology wind speed vectors
q2=ax.quiver(dat['merra2']['spd'].lon[::skip_n], dat['merra2']['spd'].lat[::skip_n],
             dat['merra2']['u'][::skip_n,::skip_n],
             dat['merra2']['v'][::skip_n,::skip_n],
             color='gray', width=w, scale=scalef, scale_units='inches', units='height',
             transform=trans, zorder=100)
qk=ax.quiverkey(q2, .97, 1.02, key_length, 'MERRA-2 clim', labelcolor='gray', labelpos='W')

ax.add_feature(cfeature.LAND, fc=(1,1,1), zorder=10)
ax.coastlines(color='k')
ax.set_extent(map_bnds, crs=trans)
gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
gl.xformatter=LONGITUDE_FORMATTER
gl.yformatter=LATITUDE_FORMATTER
gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':10}
gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':10}

 
# add colorbar
cax=fig.add_axes([1.01, 0.2, 0.02, 0.6])
cbar=fig.colorbar(cf, orientation='vertical', extend='both', cax=cax)
cbar.set_label('[m/s]', labelpad=10, rotation=270, size=10, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=10)